In [ ]:
import os
import sys

# Move up one directory to the project root and add it to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Now your imports will work perfectly!
from src.data_loader import load_insurance_data, calculate_insurance_metrics
from src.eda_utils import get_missing_summary, plot_loss_ratio_by_dimension

In [ ]:
# Execution Cell Sequence
import pandas as pd
import shap
import matplotlib.pyplot as plt
from src.data_loader import load_insurance_data
from src.modeling import prepare_insurance_features, train_and_evaluate_regression

# Load data assets
zip_path = '../data/MachineLearningRating_v3.zip' 
df = load_insurance_data(zip_path)

# Split data and target vectors
X_train, X_test, y_train, y_test = prepare_insurance_features(df, is_regression=True)

# Fit models and return comparative tables
metrics, best_xgb_model = train_and_evaluate_regression(X_train, X_test, y_train, y_test)
print(pd.DataFrame(metrics).T.to_markdown())


3. Model Interpretability with SHAP
To generate explainable outputs for management review, run the SHAP calculation sequence against your best-performing ensemble tree:

In [ ]:
# Calculate feature attributions using TreeSHAP
explainer = shap.TreeExplainer(best_xgb_model)
shap_values = explainer(X_test)

# Generate Summary Plot 
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("Risk Driver Feature Importance (SHAP values)", fontsize=14)
plt.tight_layout()
plt.savefig("../reports/shap_importance_summary.png")
plt.show()


Explaining SHAP Values to Leadership
When interpreting your SHAP plots, provide clear business context for the top features:
•	CustomValueEstimate: High asset values pull SHAP points strongly to the right. This indicates that more valuable vehicles significantly increase potential claim severity, requiring higher baseline premium levels.
•	RegistrationYear (Vehicle Age): If older vehicles shift SHAP boundaries towards higher costs, it indicates that aging vehicle components increase repair complexity, which should be accounted for using an age-based premium scalar.
